<a href="https://colab.research.google.com/github/YASH-HASH-PIXEL/-ANN-Linear_Regression/blob/main/GPT_CHATBOT_PROJECT_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this in a cell to test the chatbot directly
import os
import pickle
import random
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Get working directory
BASE_DIR = os.getcwd()
VEC_PATH = os.path.join(BASE_DIR, "vectorizer.pkl")
DATA_PATH = os.path.join(BASE_DIR, "chatbot_data.pkl")

# Create sample data
sample_patterns = [
    "hello", "hi", "hey", "good morning",
    "how are you", "how's it going",
    "bye", "goodbye", "see you later",
    "what is your name", "who are you",
    "what can you do", "help",
    "thank you", "thanks",
    "what is python", "tell me about python",
    "what is machine learning",
    "how to contact support", "support"
]

sample_tags = [
    "greeting", "greeting", "greeting", "greeting",
    "how_are_you", "how_are_you",
    "goodbye", "goodbye", "goodbye",
    "identity", "identity",
    "capabilities", "capabilities",
    "thanks", "thanks",
    "python_info", "python_info",
    "ml_info",
    "support", "support"
]

responses_dict = {
    "greeting": ["Hello! How can I help?", "Hi there!", "Hey! Ask me anything."],
    "how_are_you": ["I'm great, thanks!", "All good! How can I help?"],
    "goodbye": ["Goodbye!", "See you later!", "Take care!"],
    "identity": ["I'm a FAQ chatbot.", "Your virtual assistant."],
    "capabilities": ["I answer questions.", "I'm here to help."],
    "thanks": ["You're welcome!", "Glad to help!"],
    "python_info": ["Python is a programming language.", "Python is versatile."],
    "ml_info": ["ML is a subset of AI.", "Machine learning builds models."],
    "support": ["Contact support@example.com", "Call 1-800-HELP for support."]
}

# Create vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(sample_patterns)

# Save data
responses_list = [{tag: responses} for tag, responses in responses_dict.items()]
data = {
    "patterns": sample_patterns,
    "tags": sample_tags,
    "responses": responses_list,
    "X": X
}

with open(VEC_PATH, "wb") as f:
    pickle.dump(vectorizer, f)
with open(DATA_PATH, "wb") as f:
    pickle.dump(data, f)

print(" Chatbot data created and saved!")

# Test the chatbot
def chat(message):
    user_vec = vectorizer.transform([message])
    similarities = cosine_similarity(user_vec, X).flatten()
    best_idx = int(np.argmax(similarities))
    best_sim = similarities[best_idx]
    tag = sample_tags[best_idx]

    print(f"Similarity: {best_sim:.4f} | Tag: {tag}")

    if best_sim < 0.2:
        return "I'm not sure I understand."

    return random.choice(responses_dict[tag])

# Test messages
test_messages = [
    "hello",
    "how are you",
    "what is your name",
    "tell me about python",
    "goodbye"
]


print("TESTING CHATBOT")

for msg in test_messages:
    response = chat(msg)
    print(f"User: {msg}")
    print(f"Bot: {response}")


# Interactive mode

print("INTERACTIVE MODE")
print("Type 'quit' to exit")

while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in ['quit', 'exit', 'bye']:
        print("Bot: Goodbye!")
        break
    if user_input:
        response = chat(user_input)
        print(f"Bot: {response}")

 Chatbot data created and saved!
TESTING CHATBOT
Similarity: 1.0000 | Tag: greeting
User: hello
Bot: Hi there!
Similarity: 1.0000 | Tag: how_are_you
User: how are you
Bot: I'm great, thanks!
Similarity: 1.0000 | Tag: identity
User: what is your name
Bot: Your virtual assistant.
Similarity: 1.0000 | Tag: python_info
User: tell me about python
Bot: Python is a programming language.
Similarity: 1.0000 | Tag: goodbye
User: goodbye
Bot: Take care!
INTERACTIVE MODE
Type 'quit' to exit

You:  HO
Similarity: 0.0000 | Tag: greeting
Bot: I'm not sure I understand.

You: HI
Similarity: 1.0000 | Tag: greeting
Bot: Hey! Ask me anything.

You: BYE
Bot: Goodbye!


In [2]:
import os
import pickle
import random
import logging
import numpy as np
import sys
from flask import Flask, request, jsonify, render_template_string, Response, stream_with_context
from sklearn.metrics.pairwise import cosine_similarity
import time
import json

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Path handling (Colab compatible)
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

VEC_PATH = os.path.join(BASE_DIR, "vectorizer.pkl")
DATA_PATH = os.path.join(BASE_DIR, "chatbot_data.pkl")

#  Load or create data
def load_or_create_data():
    """Load existing data or create sample data"""
    if os.path.exists(VEC_PATH) and os.path.exists(DATA_PATH):
        logger.info("Loading existing model files...")
        with open(VEC_PATH, "rb") as f:
            vectorizer = pickle.load(f)
        with open(DATA_PATH, "rb") as f:
            data = pickle.load(f)
        return vectorizer, data
    else:
        logger.info("Creating sample data...")
        from sklearn.feature_extraction.text import TfidfVectorizer

        sample_patterns = [
            "hello", "hi", "hey", "good morning",
            "how are you", "how's it going",
            "bye", "goodbye", "see you later",
            "what is your name", "who are you",
            "what can you do", "help",
            "thank you", "thanks",
            "what is python", "tell me about python",
            "what is machine learning",
            "how to contact support", "support"
        ]

        sample_tags = [
            "greeting", "greeting", "greeting", "greeting",
            "how_are_you", "how_are_you",
            "goodbye", "goodbye", "goodbye",
            "identity", "identity",
            "capabilities", "capabilities",
            "thanks", "thanks",
            "python_info", "python_info",
            "ml_info",
            "support", "support"
        ]

        responses_dict = {
            "greeting": ["Hello! How can I help?", "Hi there!", "Hey! Ask me anything."],
            "how_are_you": ["I'm great, thanks!", "All good! How can I help?"],
            "goodbye": ["Goodbye!", "See you later!", "Take care!"],
            "identity": ["I'm a FAQ chatbot.", "Your virtual assistant."],
            "capabilities": ["I answer questions.", "I'm here to help."],
            "thanks": ["You're welcome!", "Glad to help!"],
            "python_info": ["Python is a programming language.", "Python is versatile."],
            "ml_info": ["ML is a subset of AI.", "Machine learning builds models."],
            "support": ["Contact support@example.com", "Call 1-800-HELP for support."]
        }

        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(sample_patterns)

        responses_list = [{tag: responses} for tag, responses in responses_dict.items()]

        data = {
            "patterns": sample_patterns,
            "tags": sample_tags,
            "responses": responses_list,
            "X": X
        }

        # Save for future use
        with open(VEC_PATH, "wb") as f:
            pickle.dump(vectorizer, f)
        with open(DATA_PATH, "wb") as f:
            pickle.dump(data, f)

        return vectorizer, data

# Initialize chatbot
vectorizer, data = load_or_create_data()
patterns = data["patterns"]
tags = data["tags"]
X = data["X"]

# Build responses dict
responses_dict = {}
for item in data["responses"]:
    if isinstance(item, dict):
        for tag, reply_list in item.items():
            responses_dict[tag] = reply_list

logger.info(f"Loaded {len(set(tags))} categories")

def get_response(user_input):
    """Generate response with ultra-fast lookup"""
    try:
        # Pre-process and vectorize
        user_vec = vectorizer.transform([user_input])

        # Fast similarity calculation
        similarities = cosine_similarity(user_vec, X).flatten()
        best_idx = int(np.argmax(similarities))
        best_sim = similarities[best_idx]
        tag = tags[best_idx]

        logger.info(f"Input: '{user_input}' | Sim: {best_sim:.4f} | Tag: '{tag}'")

        if best_sim < 0.2:
            return "I'm not sure I understand. Could you rephrase that?"

        # Instant response selection
        return random.choice(responses_dict.get(tag, ["I don't have an answer for that."]))
    except Exception as e:
        logger.error(f"Error: {e}")
        return "Sorry, I encountered an error."

# Flask app
app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'healthy', 'categories': list(responses_dict.keys())})

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/chat', methods=['POST'])
def chat():
    """Ultra-fast chat endpoint"""
    try:
        data = request.get_json()
        if not data or 'message' not in data:
            return jsonify({'error': 'Missing message'}), 400

        message = data['message'].strip()
        if not message:
            return jsonify({'reply': 'Please say something!'})

        # Instant response generation
        start_time = time.time()
        reply = get_response(message)
        response_time = (time.time() - start_time) * 1000  # Convert to ms

        return jsonify({
            'reply': reply,
            'response_time_ms': round(response_time, 2),
            'timestamp': time.time()
        })
    except Exception as e:
        logger.error(f"Chat error: {e}")
        return jsonify({'error': 'Internal error'}), 500

@app.route('/chat/stream', methods=['POST'])
def chat_stream():
    """Streaming chat endpoint for real-time feel"""
    def generate():
        try:
            data = request.get_json()
            message = data.get('message', '').strip()

            if message:
                # Simulate typing effect
                reply = get_response(message)
                words = reply.split()

                # Stream word by word for realistic typing effect
                for i, word in enumerate(words):
                    chunk = {
                        'text': word + ' ',
                        'complete': i == len(words) - 1,
                        'full_response': reply if i == len(words) - 1 else None
                    }
                    yield f"data: {json.dumps(chunk)}\n\n"
                    time.sleep(0.05)  # 50ms delay for natural typing feel

        except Exception as e:
            yield f"data: {json.dumps({'error': str(e)})}\n\n"

    return Response(
        stream_with_context(generate()),
        mimetype='text/event-stream',
        headers={
            'Cache-Control': 'no-cache',
            'Connection': 'keep-alive',
            'X-Accel-Buffering': 'no'
        }
    )

# Ultra-responsive HTML with real-time typing animation
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>FAQ Chatbot - Real-Time</title>
    <meta charset="UTF-8">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .chat-container {
            width: 90%;
            max-width: 600px;
            height: 80vh;
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            display: flex;
            flex-direction: column;
            overflow: hidden;
        }
        .chat-header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            text-align: center;
            font-size: 1.5em;
            font-weight: bold;
        }
        .status-indicator {
            display: inline-block;
            width: 10px;
            height: 10px;
            background: #4CAF50;
            border-radius: 50%;
            margin-right: 10px;
            animation: pulse 2s infinite;
        }
        @keyframes pulse {
            0% { box-shadow: 0 0 0 0 rgba(76, 175, 80, 0.7); }
            70% { box-shadow: 0 0 0 10px rgba(76, 175, 80, 0); }
            100% { box-shadow: 0 0 0 0 rgba(76, 175, 80, 0); }
        }
        #chat-messages {
            flex: 1;
            overflow-y: auto;
            padding: 20px;
            background: #f8f9fa;
        }
        .message {
            margin-bottom: 15px;
            animation: slideIn 0.3s ease-out;
        }
        @keyframes slideIn {
            from { opacity: 0; transform: translateY(10px); }
            to { opacity: 1; transform: translateY(0); }
        }
        .user-message {
            text-align: right;
        }
        .user-message .message-content {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 12px 18px;
            border-radius: 20px 20px 5px 20px;
            display: inline-block;
            max-width: 80%;
            word-wrap: break-word;
            box-shadow: 0 2px 10px rgba(102, 126, 234, 0.3);
        }
        .bot-message .message-content {
            background: white;
            color: #333;
            padding: 12px 18px;
            border-radius: 20px 20px 20px 5px;
            display: inline-block;
            max-width: 80%;
            word-wrap: break-word;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }
        .typing-indicator {
            display: none;
            padding: 12px 18px;
        }
        .typing-dots {
            display: inline-flex;
            gap: 5px;
        }
        .typing-dot {
            width: 8px;
            height: 8px;
            background: #667eea;
            border-radius: 50%;
            animation: typing 1.4s infinite;
        }
        .typing-dot:nth-child(2) { animation-delay: 0.2s; }
        .typing-dot:nth-child(3) { animation-delay: 0.4s; }
        @keyframes typing {
            0%, 60%, 100% { transform: translateY(0); opacity: 0.3; }
            30% { transform: translateY(-10px); opacity: 1; }
        }
        .response-time {
            font-size: 0.7em;
            color: #999;
            margin-top: 5px;
        }
        .input-container {
            padding: 20px;
            background: white;
            border-top: 1px solid #e0e0e0;
        }
        .input-group {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex: 1;
            padding: 15px;
            border: 2px solid #e0e0e0;
            border-radius: 25px;
            font-size: 16px;
            outline: none;
            transition: border-color 0.3s;
        }
        #user-input:focus {
            border-color: #667eea;
        }
        #send-button {
            width: 50px;
            height: 50px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 50%;
            cursor: pointer;
            font-size: 20px;
            transition: transform 0.2s;
        }
        #send-button:hover {
            transform: scale(1.1);
        }
        #send-button:active {
            transform: scale(0.95);
        }
        .suggestions {
            display: flex;
            gap: 8px;
            padding: 10px 20px;
            overflow-x: auto;
            background: #f8f9fa;
        }
        .suggestion-chip {
            white-space: nowrap;
            padding: 8px 15px;
            background: white;
            border: 1px solid #e0e0e0;
            border-radius: 20px;
            cursor: pointer;
            font-size: 14px;
            transition: all 0.3s;
        }
        .suggestion-chip:hover {
            background: #667eea;
            color: white;
            border-color: #667eea;
        }
    </style>
</head>
<body>
    <div class="chat-container">
        <div class="chat-header">
            <span class="status-indicator"></span>
            AI Assistant
        </div>

        <div id="chat-messages"></div>

        <div id="typing-indicator" class="typing-indicator">
            <div class="typing-dots">
                <div class="typing-dot"></div>
                <div class="typing-dot"></div>
                <div class="typing-dot"></div>
            </div>
        </div>

        <div class="suggestions" id="suggestions">
            <span class="suggestion-chip" onclick="quickMessage('Hello')">👋 Hello</span>
            <span class="suggestion-chip" onclick="quickMessage('How are you?')">💭 How are you?</span>
            <span class="suggestion-chip" onclick="quickMessage('What is Python?')">🐍 Python</span>
            <span class="suggestion-chip" onclick="quickMessage('What can you do?')">🤖 Capabilities</span>
        </div>

        <div class="input-container">
            <div class="input-group">
                <input type="text" id="user-input" placeholder="Type your message..." autocomplete="off">
                <button id="send-button" onclick="sendMessage()">➤</button>
            </div>
        </div>
    </div>

    <script>
        const chatMessages = document.getElementById('chat-messages');
        const userInput = document.getElementById('user-input');
        const sendButton = document.getElementById('send-button');
        const typingIndicator = document.getElementById('typing-indicator');

        // Focus input on load
        userInput.focus();

        // Add initial bot message
        addBotMessage("Hello! I'm your AI assistant. How can I help you today?");

        function addUserMessage(text) {
            const messageDiv = document.createElement('div');
            messageDiv.className = 'message user-message';
            messageDiv.innerHTML = `<div class="message-content">${escapeHtml(text)}</div>`;
            chatMessages.appendChild(messageDiv);
            scrollToBottom();
        }

        function addBotMessage(text, responseTime = null) {
            const messageDiv = document.createElement('div');
            messageDiv.className = 'message bot-message';
            let html = `<div class="message-content">${escapeHtml(text)}</div>`;
            if (responseTime) {
                html += `<div class="response-time">⚡ ${responseTime}ms</div>`;
            }
            messageDiv.innerHTML = html;
            chatMessages.appendChild(messageDiv);
            scrollToBottom();
        }

        function updateLastBotMessage(text) {
            const messages = chatMessages.getElementsByClassName('bot-message');
            if (messages.length > 0) {
                const lastMessage = messages[messages.length - 1];
                const contentDiv = lastMessage.querySelector('.message-content');
                contentDiv.textContent = text;
                scrollToBottom();
            }
        }

        function scrollToBottom() {
            chatMessages.scrollTop = chatMessages.scrollHeight;
        }

        function escapeHtml(text) {
            const div = document.createElement('div');
            div.textContent = text;
            return div.innerHTML;
        }

        function showTypingIndicator() {
            typingIndicator.style.display = 'block';
            scrollToBottom();
        }

        function hideTypingIndicator() {
            typingIndicator.style.display = 'none';
        }

        async function sendMessage(message = null) {
            const text = message || userInput.value.trim();
            if (!text) return;

            // Add user message
            addUserMessage(text);

            // Clear input
            if (!message) {
                userInput.value = '';
            }

            // Show typing indicator
            showTypingIndicator();

            // Disable input
            userInput.disabled = true;
            sendButton.disabled = true;

            try {
                const startTime = performance.now();

                const response = await fetch('/chat', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ message: text })
                });

                const data = await response.json();
                const endTime = performance.now();
                const responseTime = Math.round(endTime - startTime);

                // Hide typing indicator
                hideTypingIndicator();

                if (data.error) {
                    addBotMessage('Sorry, I encountered an error. Please try again.');
                } else {
                    // Add bot response with typing animation
                    await typeWriterEffect(data.reply, responseTime);
                }
            } catch (err) {
                hideTypingIndicator();
                addBotMessage('Connection error. Please check your network and try again.');
            } finally {
                // Re-enable input
                userInput.disabled = false;
                sendButton.disabled = false;
                userInput.focus();
            }
        }

        async function typeWriterEffect(text, responseTime) {
            const messageDiv = document.createElement('div');
            messageDiv.className = 'message bot-message';
            const contentDiv = document.createElement('div');
            contentDiv.className = 'message-content';
            messageDiv.appendChild(contentDiv);
            chatMessages.appendChild(messageDiv);

            // Type out each character
            for (let i = 0; i < text.length; i++) {
                contentDiv.textContent += text[i];
                scrollToBottom();
                await new Promise(resolve => setTimeout(resolve, 20)); // Speed of typing
            }

            // Add response time
            if (responseTime) {
                const timeDiv = document.createElement('div');
                timeDiv.className = 'response-time';
                timeDiv.textContent = `⚡ ${responseTime}ms`;
                messageDiv.appendChild(timeDiv);
            }
        }

        function quickMessage(text) {
            sendMessage(text);
        }

        // Event listeners
        userInput.addEventListener('keypress', (e) => {
            if (e.key === 'Enter' && !e.shiftKey) {
                e.preventDefault();
                sendMessage();
            }
        });

        // Keep connection alive
        setInterval(async () => {
            try {
                await fetch('/health');
            } catch (err) {
                console.log('Health check failed');
            }
        }, 30000);
    </script>
</body>
</html>
"""

if __name__ == '__main__':
    from threading import Thread

    # Optimize Flask for speed
    app.config['SEND_FILE_MAX_AGE_DEFAULT'] = 0
    app.config['TEMPLATES_AUTO_RELOAD'] = False

    # Run Flask in a separate thread for Colab
    t = Thread(target=app.run, kwargs={
        'host': '0.0.0.0',
        'port': 5000,
        'threaded': True,  # Enable threading for concurrent requests
        'use_reloader': False
    })
    t.daemon = True
    t.start()

    # Display clickable link for Colab
    from google.colab.output import eval_js

    print("Real-Time FAQ Chatbot is LIVE!")

    print("Click the link below to open the chatbot:")
    print(eval_js("google.colab.kernel.proxyPort(5000)"))


Real-Time FAQ Chatbot is LIVE!
Click the link below to open the chatbot:
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


https://5000-m-s-kkb-use4a1-2ssvagq1o4g6y-a.us-east4-1.prod.colab.dev
